In [1]:
# Main to test data train size influence on model predictions

In [2]:
nr = '005'
RandSeedNumber = 80234
train_size = 80

In [3]:
import os
import numpy as np
import torch
torch.cuda.empty_cache()
import torch_geometric
from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader 
from GNNmodel_Model_BatchNorm_EdgeAttr import myPNA
from GNNmodel_Visuals import visualize_snapshot_cell, visualize_snapshot_nucleus, visualize_snapshot_graph, lin_regression
import scipy.stats
from scipy.stats import linregress
import seaborn as sns
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy import io
import pickle
import random
import copy
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
plt.rcParams['font.family'] = 'serif'
print(torch.cuda.is_available())

plt.rcParams['xtick.direction'] = plt.rcParams['ytick.direction'] = 'in'

/home/20234017/.local/lib/python3.11/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /vast.mnt/home/20234017/.local/lib/python3.11/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs
  import torch_geometric.typing


True


In [4]:
# function for loading the processed data

def load_dataset(file_name):
    with open(file_name, 'rb') as f:
        data = pickle.load(f)
        return data

In [5]:
# load data
data_list_all = load_dataset('AllProcessedData_BigGrid_batch3_[FINAL].pkl')
data_list_validation = load_dataset('AllProcessedData_BigGrid_batch3_[FINAL_VALIDATION].pkl')

In [6]:
# Reduce amount of low D values by only taking 2 simulations from each of those setting pairs

selected_indices = {
    0, 1, 2,
    9, 10, 11,
    18, 19, 20,
    27, 28, 29,
    36, 37, 38,
    45, 46, 47,
    54, 55, 56, 57, 58,
    63, 64, 65, 66, 67, 68,
    72, 73, 74, 75, 76, 77, 78, 79}

filtered_data_list = []

for block_idx in range(81):
    start = block_idx * 12
    end = start + 12

    block = data_list_all[start:end]

    if block_idx in selected_indices:
        filtered_data_list.extend(block[10:])  # keep last 2
    else:
        filtered_data_list.extend(block)      # keep all 12

print(f"Original length: {len(data_list_all)}")
print(f"Filtered length: {len(filtered_data_list)}")

Original length: 972
Filtered length: 602


In [7]:
# function for setting random seed number

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # GPU will be slower but deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False 
    print("Seeded everything: {}".format(seed))

In [8]:
# Get the statistics to normalize the input

def compute_stats(data_list):
    cell_area = []
    cell_perim = []
    cell_si = []
    nuc_area = []
    nuc_perim = []
    nuc_si = []
    diff = []
    edge_attr = []
    x_pos = []
    y_pos = []

    for data in data_list:
        cell_area.extend(data.x[:, 3])
        cell_perim.extend(data.x[:, 4])
        cell_si.extend(data.x[:, 5])
        nuc_area.extend(data.x[:, 0])
        nuc_perim.extend(data.x[:, 1])
        nuc_si.extend(data.x[:, 2])
        diff.extend([data.diffConst.item()])
        edge_attr.extend(data.edge_attr)
        x_pos.extend(data.cell_pos[:, 0])
        y_pos.extend(data.cell_pos[:, 1])

    stats = {
        "cell_area": (np.mean(cell_area), np.std(cell_area)),
        "cell_perim": (np.mean(cell_perim), np.std(cell_perim)),
        "cell_si": (np.mean(cell_si), np.std(cell_si)),
        "nuc_area": (np.mean(nuc_area), np.std(nuc_area)),
        "nuc_perim": (np.mean(nuc_perim), np.std(nuc_perim)),
        "nuc_si": (np.mean(nuc_si), np.std(nuc_si)),
        "diff": (np.mean(diff), np.std(diff)),
        "edge_attr": (np.mean(edge_attr), np.std(edge_attr)),
        "x_pos": (np.mean(x_pos), np.std(x_pos)),
        "y_pos": (np.mean(y_pos), np.std(y_pos)),
    }
    return stats

In [9]:
# function for picking diffusion constant for given snapshot as output

def setOutput(data_list):
    for data in data_list:
        data.y = data.diffConst
    return data_list

In [10]:
# function for choosing the node and edge information as the input

def setInput(data_list):
    for data in data_list:
        data.x = data.x
    return data_list

In [11]:
# Function to normalize all input data 

def normalize(data_list, stats):
    for data in data_list:
        
        # x features
        data.x[:, 3] = (data.x[:, 3] - stats["cell_area"][0]) / stats["cell_area"][1]
        data.x[:, 4] = (data.x[:, 4] - stats["cell_perim"][0]) / stats["cell_perim"][1]
        data.x[:, 5] = (data.x[:, 5] - stats["cell_si"][0]) / stats["cell_si"][1]

        data.x[:, 0] = (data.x[:, 0] - stats["nuc_area"][0]) / stats["nuc_area"][1]
        data.x[:, 1] = (data.x[:, 1] - stats["nuc_perim"][0]) / stats["nuc_perim"][1]
        data.x[:, 2] = (data.x[:, 2] - stats["nuc_si"][0]) / stats["nuc_si"][1]

        # other features
        data.diffConst = data.diffConst / stats["diff"][1]
        data.edge_attr = (data.edge_attr - stats["edge_attr"][0]) / stats["edge_attr"][1]
        data.cell_pos[:,0] = (data.cell_pos[:,0] - stats["x_pos"][0]) / stats["x_pos"][1]
        data.cell_pos[:,1] = (data.cell_pos[:,1] - stats["y_pos"][0]) / stats["y_pos"][1]

    return data_list

In [12]:
# Function to train the model and input area fraction as graph level

def train():
    model.train()
    for data in train_loader:
        optimizer.zero_grad()

        edge_attr = data.edge_attr.view(-1, 1)
        
        # calculate loss
        pred = model(data.x.cuda(), data.edge_index.cuda(), data.edge_attr.cuda(), data.batch.cuda(), data.area_frac.cuda())
        loss = criterion(pred.squeeze(), data.y.squeeze().cuda())

        # update
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    return pred.squeeze().detach().cpu(), data.y.squeeze()

In [13]:
@torch.no_grad()
def test(loader):
    model.eval()
    loss_list=[]
    pred_list=[]
    y_list=[]
    for data in loader:
        edge_attr = data.edge_attr.view(-1, 1)
        pred = model(data.x.cuda(), data.edge_index.cuda(), data.edge_attr.cuda(), data.batch.cuda(), data.area_frac.cuda())
        loss = criterion(pred.squeeze().detach().cpu(), data.y.squeeze())
        loss_list.append(loss.item())
        pred_list.extend(pred.squeeze().detach().cpu())
        y_list.extend(data.y.squeeze())
    return loss_list, pred_list, y_list

In [14]:
# Function to randomly select test and train indices from the 30 simulations out 300

def select_indices(train_size):

    selected_idx = np.random.choice(np.arange(len(filtered_data_list)), size=len(filtered_data_list), replace=False)
    test_idx = np.random.choice(selected_idx, size=int(0.2*len(selected_idx)), replace=False)
    
    remaining_idx = np.setdiff1d(selected_idx, test_idx)
    train_idx = np.random.choice(remaining_idx, size=int(train_size*len(selected_idx)), replace=False)
    val_idx1 = selected_idx

    val_idx = np.random.choice(np.arange(972), size=972, replace=False)

    test_idx_list = test_idx.tolist()
    train_idx_list = train_idx.tolist()
    val_idx_list1 = val_idx1.tolist()
    val_idx_list = val_idx.tolist()

    return test_idx_list, train_idx_list, val_idx_list1, val_idx_list

In [15]:
# MAIN CODE WHERE TRAINING ACTUALLY HAPPENS

In [16]:
# create folders for saving results
os.makedirs('TrainingResults_batch3_gridBig_[TrainSize]', exist_ok=True)
os.makedirs('TrainingWeights_batch3_gridBig_[TrainSize]', exist_ok=True)

In [17]:
# rand seed initialization
set_seed(RandSeedNumber)

Seeded everything: 80234


In [18]:
# Select the indices of the simulations used for training and testing

test_idx, train_idx, val_idx1, val_idx = select_indices(train_size/100)

In [19]:
# Create the list with the actual data to train and test with

data_list_train = []
data_list_test = []
data_list_val = []

for i, item in enumerate(filtered_data_list):  # 602 folders
    if i in test_idx:
        data_list_test.extend(item)        
    if i in train_idx:
        data_list_train.extend(item)

for i, item in enumerate(data_list_validation):  # 972 folders
    if i in val_idx:
        data_list_val.extend(item)   

print(f'Train size:{len(data_list_train)}, Test size:{len(data_list_test)}, Validation size:{len(data_list_val)}')

Train size:2405, Test size:600, Validation1 size:3010, Validation size:2916


In [21]:
# Compute mean and std for all features

stats = compute_stats(data_list_train)

In [22]:
# nondimesionalize lists

data_train_nondim = normalize(data_list_train, stats)
data_test_nondim = normalize(data_list_test, stats)
data_val_nondim = normalize(data_list_val, stats)

In [23]:
# Visualize the normalized diffusion constant distribution of the train data

D_norm_train = []
for data in data_train_nondim:
    D_norm_train.append(data.diffConst)

plt.hist(D_norm_train, bins=10, edgecolor='black')
plt.xlabel("D")
plt.ylabel("Counts")

plt.savefig(f"TrainingResults_batch3_gridBig_[TrainSize]/Hist_DnormTrain_[oldredo{train_size}]_{nr}.png", dpi=200, bbox_inches="tight")
plt.close()

In [24]:
# Shuffle the content of both lists since now all snapshots from same simulation are clustered together

random.shuffle(data_train_nondim)
random.shuffle(data_test_nondim)
random.shuffle(data_val_nondim)

In [25]:
# Select the diffusion constant as the output

data_train_nondim = setOutput(data_train_nondim)
data_test_nondim = setOutput(data_test_nondim)
data_val_nondim = setOutput(data_val_nondim)

In [26]:
# Select the node features and edges as the input 

data_train_nondim = setInput(data_train_nondim)
data_test_nondim = setInput(data_test_nondim)
data_val_nondim = setInput(data_val_nondim) 

In [27]:
# Visualize snapshots and graph representation of data

visualize_snapshot_cell(data_val_nondim, [50, 100])
visualize_snapshot_nucleus(data_val_nondim, [50, 100])
visualize_snapshot_graph(data_val_nondim, [50, 100])

In [28]:
# Settings for the model - MAIN VARIABLES

num_layers      = 5     
hidden_channels = 24     
batch_size      = 64   
learning_rate   = 1e-5   
weight_decay    = 1e-3   
EpochNum        = 151 

In [29]:
# Test what the capabilities of a linear regression model are

corr, MSE, p = lin_regression(data_list_val)
print(f'The correlation is: {corr} and the MSE is: {MSE}')

The correlation is: -0.4226669235673423 and the MSE is: 0.7087199778823895


In [ ]:
g = torch.Generator()
g.manual_seed(RandSeedNumber)

In [30]:
# Use Dataloader to make data ready to train model with

train_loader = DataLoader(data_train_nondim, batch_size=batch_size, shuffle=True, generator=g, drop_last=True)
test_loader = DataLoader(data_test_nondim, batch_size=4096, shuffle=False, generator=g, drop_last=False) 

In [31]:
# initialize model

in_channels = data_list_train[0].x.shape[1] # Shape has (num_nodes, features) and you only want features
model = myPNA(data_list=data_train_nondim, in_channels=in_channels, hidden_channels=hidden_channels, num_layers=num_layers).cuda()
print(model)

tensor([     0,    547,   6473,  25609,  34149,  45201, 106638,  21212,    670,
             1])
myPNA(
  (convs): ModuleList(
    (0): PNAConv(6, 24, towers=3, edge_dim=1)
    (1-4): 4 x PNAConv(24, 24, towers=3, edge_dim=1)
  )
  (batch_norms): ModuleList(
    (0-4): 5 x BatchNorm(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (lin): Linear(in_features=25, out_features=1, bias=True)
)


In [32]:
# initialize optimizer

criterion = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
use_amp = True
scaler = torch.cuda.amp.GradScaler(enabled=use_amp) 

In [34]:
# train model
train_loss_lst=[]
test_loss_lst=[]
train_corr_lst=[]
test_corr_lst=[]
for epoch in range(1, EpochNum):
    pred, truth = train()

    with torch.no_grad():
        train_loss, train_pred, train_truth = test(train_loader)
        test_loss, test_pred, test_truth = test(test_loader)
        train_loss_lst.append(sum(train_loss)/len(train_loss))
        test_loss_lst.append(sum(test_loss)/len(test_loss))

        # calculate correlation
        train_corr, train_p_value = scipy.stats.pearsonr(np.array(train_pred),np.array(train_truth))
        test_corr, test_p_value = scipy.stats.pearsonr(np.array(test_pred),np.array(test_truth))

        train_corr_lst.append(train_corr)
        test_corr_lst.append(test_corr)

        print(f'Epoch: {epoch:03d}, Train loss: {sum(train_loss)/len(train_loss):.4f}, Test loss: {sum(test_loss)/len(test_loss):.4f},\
            Pearson correlation (train): {train_corr:.4f}, Pearson correlation (test): {test_corr:.4f}')

/home/20234017/.local/lib/python3.11/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/home/20234017/.local/lib/python3.11/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='min')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch: 001, Train loss: 1.1462, Test loss: 0.8071,            Pearson correlation (train): 0.6334, Pearson correlation (test): 0.6538
Epoch: 002, Train loss: 1.0178, Test loss: 0.7170,            Pearson correlation (train): 0.6711, Pearson correlation (test): 0.6468
Epoch: 003, Train loss: 0.9117, Test loss: 0.6401,            Pearson correlation (train): 0.7238, Pearson correlation (test): 0.6853
Epoch: 004, Train loss: 0.8062, Test loss: 0.5825,            Pearson correlation (train): 0.7601, Pearson correlation (test): 0.7161
Epoch: 005, Train loss: 0.7525, Test loss: 0.5420,            Pearson correlation (train): 0.7871, Pearson correlation (test): 0.7413
Epoch: 006, Train loss: 0.7316, Test loss: 0.5288,            Pearson correlation (train): 0.8059, Pearson correlation (test): 0.7626
Epoch: 007, Train loss: 0.7042, Test loss: 0.5055,            Pearson correlation (train): 0.8189, Pearson correlation (test): 0.7775
Epoch: 008, Train loss: 0.6575, Test loss: 0.4804,            

In [35]:
# POST PROCESSING AND VISUALIZATION

In [36]:
# Plot the loss curves

def plot_losses(train, test):
    plt.figure()
    plt.semilogy(train, label="train", marker=".", lw=2)
    plt.semilogy(test, label="test", marker=".", lw=2)
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.legend()
    plt.savefig(f"TrainingResults_batch3_gridBig_[TrainSize]/losses_[trainsize{train_size}]_{RandSeedNumber}_{nr}.png", dpi=200)
    plt.close()

plot_losses(train_loss_lst, test_loss_lst)

In [37]:
# Test the model on validation data (all the data)

val_loader = DataLoader(data_val_nondim, batch_size=2980, shuffle=False, generator=g, drop_last=False) 
val_loss, val_pred, val_truth = test(val_loader)

In [38]:
# Scale the model predictions back according to pre-processing

truth = np.array(val_truth) * stats["diff"][1] 
pred  = np.array(val_pred) * stats["diff"][1] 

In [39]:
print(pred.min())
print(truth.min())

0.048020996
0.011749101


In [40]:
# Obtain Pearson correlation of the validation dataset

val_corr, val_p_value = scipy.stats.pearsonr(np.array(pred),np.array(truth))
print(f'The Pearson correlation is: {val_corr}')
print(f'The p-value is: {val_p_value}')

The Pearson correlation is: 0.9272185365677742
The p-value is: 0.0


In [42]:
# obtain least-squares fit parameters

m, b = np.polyfit(truth, pred, 1)
print(f'The slope is: {m} and the intercept is: {b}')

The slope is: 0.613619648342066 and the intercept is: 0.32228374646846175


In [43]:
# 2D probability density scatter plot of predictions

plt.figure(figsize=(4, 3))
ax = sns.kdeplot(x=np.array(truth), y=np.array(pred), cmap="rocket_r", fill=True, levels=10, cbar=True, cut=12)
plt.scatter(np.array(truth),np.array(pred), s=5, c='lightblue', alpha=0.6, label = 'Data')

# reference line
xref = np.linspace(0, 6.0, num=100)
plt.plot(xref,xref, c = 'k', label = 'Reference',linewidth = 1, linestyle = 'dashed')

# plot line with intercept
x = np.linspace(0, 6, 100)
y2 = m*x + b
plt.plot(x,y2, label = "Line", linewidth = 1)

plt.xlim([0,6.0])
plt.ylim([0,6.0])
plt.xlabel('$100D$',fontsize=12)
plt.ylabel('$100D_{NN}$',fontsize=12)
plt.tight_layout()
plt.savefig(f'TrainingResults_batch3_gridBig_[TrainSize]/Fit2_Line_noLog_[trainsize{train_size}]_{m:.3f}_{b:.3f}_{RandSeedNumber}_{nr}.png', dpi=300, bbox_inches='tight')
plt.close()

In [44]:
# save results

savePath = f'TrainingResults_batch3_gridBig_[TrainSize]/GNNmodel_redo2_[trainsize{train_size}]_{nr}_layers{num_layers}_channels{hidden_channels}_batchSize{batch_size}_lr{learning_rate}_Epochs{EpochNum}_WeightDecay{weight_decay}_seed{RandSeedNumber}.mat'
io.savemat(savePath, {'train_loss': train_loss_lst,  'train_corr': train_corr_lst,
                      'test_loss' : test_loss_lst,   'test_corr': test_corr_lst})
print(savePath)

TrainingResults_batch3_gridBig_[TrainSize]/GNNmodel_redo2_[trainsize80]_005_layers5_channels24_batchSize64_lr1e-05_Epochs151_WeightDecay0.001_seed80234.mat


In [45]:
# save trained model weights

torch.save(model.state_dict(), f'TrainingWeights_batch3_gridBig_[TrainSize]/GNNmodel_weights_redo2_[trainsize{train_size}]_{RandSeedNumber}_{nr}.pth')

In [46]:
# plotting the resulting average Pearson values and standard error of mean

plt.figure()

x = np.array([20, 30, 40, 50, 60, 70, 80])
values = np.array([0.9127, 0.9161, 0.9244, 0.9274, 0.9279, 0.9251, 0.9322])
std = np.array([0.0079, 0.0038, 0.0014, 0.0027, 0.0029, 0.0034, 0.0027])

# plot
plt.figure()
plt.errorbar(x, values, yerr=std, fmt='o', capsize=5)

plt.xlabel("train data size (%)")
plt.ylabel("Pearson correlation")
plt.xticks(x)
plt.grid(True, which='both', linestyle='-', linewidth=0.5, alpha=0.3)

plt.savefig(f'TrainingResults_batch3_gridBig_[TrainSize]/TrainSizeResult_5seeds_0001.png', dpi=300, bbox_inches='tight')
plt.close()

In [48]:
# result including a fit

x = np.array([20, 30, 40, 50, 60, 70, 80])
values = np.array([0.9127, 0.9161, 0.9244, 0.9274, 0.9279, 0.9251, 0.9322])
std = np.array([0.0079, 0.0038, 0.0014, 0.0027, 0.0029, 0.0034, 0.0027])

# fit
res = linregress(x, values)
a, b = res.slope, res.intercept

# line
x_line = np.linspace(x.min(), x.max(), 200)
y_line = a * x_line + b

# plot
plt.figure(figsize=(6, 4))

plt.errorbar(x, values, yerr=std, fmt='o', capsize=4, label="data")

plt.plot(
    x_line,
    y_line,
    color='green',
    linewidth=1,
    label=f"fit: y = {a:.4f}x + {b:.4f}\n$R^2$ = {res.rvalue**2:.4f}"
)

plt.xlabel("Train data size (%)")
plt.ylabel("Pearson correlation")
#plt.title("Effect of train data size on model performance")

plt.legend(frameon=False)

plt.minorticks_on()
plt.grid(True, which='major', linewidth=0.5, alpha=0.3)
plt.grid(True, which='minor', linewidth=0.3, alpha=0.15)

plt.tight_layout()

plt.savefig("TrainingResults_batch3_gridBig_[TrainSize]/TrainSizeResult_5seeds_report001.png", dpi=300, bbox_inches="tight")
plt.close()

In [50]:
# average least squares fits on pearson correlation scale

import matplotlib as mpl


slopes = np.array([
    [0.562708888, 0.500349028, 0.558292445, 0.650824576, 0.448325233],
    [0.713300544, 0.601125256, 0.596857096, 0.505479183, 0.505577348],
    [0.594401720, 0.540384994, 0.700015264, 0.607108443, 0.577771358],
    [0.589606162, 0.570786362, 0.645788362, 0.560073538, 0.685262011],
    [0.604656240, 0.573290080, 0.648293746, 0.600951806, 0.722068865],
    [0.578992084, 0.671070226, 0.704518334, 0.548018714, 0.541100292],
    [0.658063489, 0.582127047, 0.666839642, 0.744188437, 0.613619648]
])

intercepts = np.array([
    [0.560198498, 0.539945425, 0.421722306, 0.389116451, 0.430794185],
    [0.457166404, 0.406260640, 0.438754452, 0.300542087, 0.300174337],
    [0.481179708, 0.324710645, 0.422237130, 0.386863941, 0.382022704],
    [0.352811737, 0.423826705, 0.354698988, 0.341751055, 0.332460430],
    [0.298677585, 0.573290080, 0.377678193, 0.378199810, 0.443619495],
    [0.331718974, 0.328480973, 0.340607336, 0.359226695, 0.320382361],
    [0.325005241, 0.308289212, 0.341929555, 0.304461930, 0.322283746]
])

values = np.array([0.9127, 0.9161, 0.9244, 0.9274, 0.9279, 0.9251, 0.9322])

norm = mpl.colors.Normalize(vmin=values.min(), vmax=values.max())
cmap = plt.cm.viridis


# reshape: 10 groups × 5 seeds
slopes_g = slopes.reshape(7, 5)
intercepts_g = intercepts.reshape(7, 5)

mean_slopes = slopes_g.mean(axis=1)
mean_intercepts = intercepts_g.mean(axis=1)

labels = ["20%", "30%", "40%", "50%", "60%", "70%", "80%"]

x = np.linspace(0,6,100)

plt.figure(figsize=(5, 4))

fig, ax = plt.subplots(figsize=(5, 4))

for i, (m, b, r) in enumerate(zip(
        mean_slopes, mean_intercepts,
        values)):

    color = cmap(norm(r))

    y_mean = m * x + b

    ax.plot(x, y_mean, color=color, linewidth=0.8, label=labels[i])

sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  

cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label("Pearson correlation")

# reference line
xref = np.linspace(0, 6.0, 100)
plt.plot(xref, xref, c='k', linestyle='dashed', linewidth=1, label='Reference')

plt.grid(True, which='major', linestyle='-', linewidth=1.0, alpha=0.3)
plt.minorticks_on()
plt.grid(True, which='minor', linestyle=':', linewidth=0.7, alpha=0.15)

plt.xlim([0, 6.0])
plt.ylim([0, 6.0])

plt.xlabel(r'$100D$', fontsize=12)
plt.ylabel(r'$100D_{NN}$', fontsize=12)

plt.legend(
    fontsize=8,
    labelspacing=0.2,
    handlelength=1.2,
    handletextpad=0.4,
    ncol=2)

plt.tight_layout()
plt.savefig(
    'TrainingResults_batch3_gridBig_[TrainSize]/AllFits_TSize_[correctAx5seeds]_002.png',
    dpi=300,
    bbox_inches='tight')

plt.close()

In [52]:
# statistical tests to see significance of results 

from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

g1 = [0.933437756, 0.898170688, 0.919380095, 0.921683589, 0.890669598]
g2 = [0.917176969, 0.92863988, 0.918304368, 0.908136717, 0.908216484]
g3 = [0.923201637, 0.922868895, 0.930077513, 0.923543318, 0.922345936]
g4 = [0.922125541, 0.923844178, 0.932275901, 0.923469525, 0.93528498]
g5 = [0.925506233, 0.920950018, 0.928835327, 0.926118492, 0.938295266]
g6 = [0.916008244, 0.930783238, 0.93312086, 0.927382982, 0.918323796]
g7 = [0.934844526, 0.925526769, 0.933114711, 0.940305816, 0.927218537]

# ANOVA
F, p = f_oneway(g1, g2, g3, g4, g5, g6, g7)

print("ANOVA:")
print("F =", F)
print("p =", p)

# TUKEY HSD
data = g1 + g2 + g3 + g4 + g5 + g6 +g7
labels = (
    ["20%"]*5 +
    ["30%"]*5 +
    ["40%"]*5 +
    ["50%"]*5 +
    ["60%"]*5 +
    ["70%"]*5 +
    ["80%"]*5
)

tukey = pairwise_tukeyhsd(endog=data, groups=labels, alpha=0.05)
print(tukey)

ANOVA:
F = 2.9492564049078713
p = 0.023318194427635715
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
   20%    30%   0.0034 0.9962 -0.0146 0.0215  False
   20%    40%   0.0117 0.3996 -0.0063 0.0298  False
   20%    50%   0.0147  0.167 -0.0033 0.0328  False
   20%    60%   0.0153 0.1392 -0.0028 0.0333  False
   20%    70%   0.0125 0.3319 -0.0056 0.0305  False
   20%    80%   0.0195 0.0272  0.0015 0.0376   True
   30%    40%   0.0083 0.7638 -0.0097 0.0263  False
   30%    50%   0.0113 0.4436 -0.0067 0.0293  False
   30%    60%   0.0118  0.389 -0.0062 0.0299  False
   30%    70%    0.009 0.6906  -0.009 0.0271  False
   30%    80%   0.0161 0.1037 -0.0019 0.0341  False
   40%    50%    0.003 0.9982  -0.015  0.021  False
   40%    60%   0.0035 0.9955 -0.0145 0.0216  False
   40%    70%   0.0007    1.0 -0.0173 0.0188  False
   40%    80%   0.0078 0.8121 -0.0102 0.0258  False
   50%   